# Lecture 7 — Class Exercise
## Heatmap & Waterfall: Netflix Catalogue

> **Push to:** `week07/lecture07_exercise.ipynb`

**Rules:**
1. Heatmap: colour scale must match the data type (sequential for counts, diverging for above/below)
2. Waterfall: use green for additions, red for subtractions, blue for totals
3. Insight title tells the setup-conflict-resolution story (or at minimum states the finding)
4. Annotate at least one cell or bar directly

---


In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

df = pd.read_csv('../data/netflix_catalogue.csv')
print(f"Loaded: {len(df)} titles")
print(df['type'].value_counts())
print(df.head())


Loaded: 3000 titles
type
Movie      1974
TV Show    1026
Name: count, dtype: int64
      type  release_year  added_year             genre        country rating  \
0    Movie          2014        2016  Sci-Fi & Fantasy         France  PG-13   
1    Movie          2010        2014     Documentaries  United States  TV-MA   
2  TV Show          2011        2012     Kids & Family  United States  TV-14   
3    Movie          2016        2018             Anime          India     PG   
4    Movie          2014        2016     Kids & Family         Canada  TV-MA   

   duration  
0       157  
1       127  
2         6  
3       134  
4        77  


In [2]:
print("Genres:", df['genre'].value_counts().head(8))
print("\nCountries:", df['country'].value_counts().head(8))
print("\nRatings:", df['rating'].value_counts())


Genres: genre
Sports                244
Sci-Fi & Fantasy      213
Kids & Family         209
Crime                 206
Drama                 204
Horror                199
Action & Adventure    198
Thrillers             195
Name: count, dtype: int64

Countries: country
United States     932
India             337
United Kingdom    261
Japan             187
France            176
Canada            164
South Korea       151
Mexico            138
Name: count, dtype: int64

Ratings: rating
TV-MA    840
TV-14    733
PG-13    589
R        312
PG       196
TV-PG    128
G         92
TV-Y7     57
TV-G      53
Name: count, dtype: int64


## Task 1 — Heatmap: content by rating and release decade

**What to build:** A heatmap showing the number of titles by **content rating** (y-axis) and **decade** (x-axis).

**Requirements:**
- Create a 'decade' column: `df['decade'] = (df['release_year'] // 10 * 10).astype(str) + 's'`
- Filter to TV-14, TV-MA, PG-13, R, PG only (most common ratings)
- Sequential colour scale (Blues)
- Values shown in cells (`text_auto=True`)
- Insight title about which rating dominates which decade


In [3]:
import plotly.io as pio
pio.renderers.default = "notebook_connected"

In [5]:
# Task 1- Heatmap: content by rating and release decade
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

df = pd.read_csv('../data/netflix_catalogue.csv')

# Create decade column
df['decade'] = (df['release_year'] // 10 * 10).astype(str) + 's'

# Filter to the five most common ratings
ratings_keep = ['TV-14', 'TV-MA', 'PG-13', 'R', 'PG']
df_filtered = df[df['rating'].isin(ratings_keep)]

# Build pivot: rows = rating, columns = decade
pivot = (df_filtered
         .groupby(['rating', 'decade'])
         .size()
         .reset_index(name='count')
         .pivot(index='rating', columns='decade', values='count')
         .fillna(0)
         .astype(int))

# ── Plotly Express base chart ──────────────────────────────────────────────────
fig = px.imshow(
    pivot,
    color_continuous_scale='Blues',   # sequential: low = light, high = dark
    text_auto=True,                   # show count in every cell
    aspect='auto',
    height=450, width=750,
    labels={'color': 'Titles', 'x': 'Decade', 'y': 'Rating'}
)

# ── Styling ────────────────────────────────────────────────────────────────────
fig.update_traces(
    textfont=dict(size=13),
    hovertemplate='<b>%{y}</b> — %{x}<br>Titles: %{z}<extra></extra>',
)

# Annotate the single dominant cell: TV-MA in 2010s (359 titles)
fig.add_annotation(
    x='2010s', y='TV-MA',
    text='Peak: 359',
    showarrow=True,
    arrowhead=2,
    arrowcolor='#e63946',
    font=dict(color='#e63946', size=12, family='Arial'),
    ax=60, ay=-40
)

fig.update_layout(
    title=dict(
        text='TV-MA content surged in the 2010s — mature content now defines Netflix',
        font=dict(size=14, family='Arial')
    ),
    coloraxis_showscale=False,        # cell values already encode magnitude
    margin=dict(l=80, r=40, t=70, b=60),
    xaxis=dict(title='Decade', tickfont=dict(size=12)),
    yaxis=dict(title='Content Rating', tickfont=dict(size=12)),
    font=dict(family='Arial', size=11),
    plot_bgcolor='white'
)

fig.show()

## Task 2 — Waterfall: Movie vs TV Show additions by year

**What to build:** A waterfall chart showing how Netflix's **Movie library** grew year by year (2015-2022).

**Requirements:**
- Filter to Movies only
- Group by `added_year`, count titles per year
- Final bar should be the cumulative total
- Green bars (additions), blue total
- Annotation on the year with the largest single addition
- Insight title naming the growth story


In [6]:
# Task 2 — Waterfall: Movie additions to Netflix by year (2015-2022)

movies = df[df['type'] == 'Movie']

adds = (movies
        .groupby('added_year')
        .size()
        .reset_index(name='count'))

adds = adds[(adds['added_year'] >= 2015) & (adds['added_year'] <= 2022)].copy()

x_vals = adds['added_year'].astype(str).tolist() + ['Total 2015-2022']
y_vals = adds['count'].tolist() + [int(adds['count'].sum())]
measures = ['relative']*len(adds) + ['total']

# ── Waterfall trace ────────────────────────────────────────────────────────────

trace = go.Waterfall(
    x=list(range(len(x_vals))),      # 0,1,2,...8 — NOT year strings
    y=y_vals,
    measure=measures,
    connector=dict(line=dict(color='#AAAAAA', dash='dot')),
    increasing=dict(marker_color='#70AD47'),
    totals=dict(marker_color='#2E75B6'),
    texttemplate='%{y:,}',
    textposition='outside'
)

fig = go.Figure(data=[trace])
fig.update_layout(
    title='Netflix library grew more than 10x in 9 years',
    plot_bgcolor='white', paper_bgcolor='white',
    font=dict(family='Arial', size=12),
    yaxis=dict(gridcolor='#EEEEEE', title='Titles Added'),
    xaxis=dict(
        title='Year', showgrid=False,
        tickmode='array',
        tickvals=list(range(len(x_vals))),
        ticktext=x_vals              # labels shown on axis
    ),
    margin=dict(l=60, r=40, t=55, b=40),
    showlegend=False,
    height=700,
    annotations=[dict(
        x=1, y=93,                   # x=1 = index of '2016'
        text='<b>Peak year</b>',
        showarrow=True, arrowhead=2,
        arrowcolor='#e63946',
        font=dict(color='#e63946', size=11, family='Arial'),
        ax=0, ay=-40
    )]
)
fig.show()